In [1]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "12")
os.environ.setdefault("MKL_NUM_THREADS", "12")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "12")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "12")

'12'

In [2]:
from pyscf import dft, gto
import scipy
import numpy as np
from XTDDFT_dev.XTDDFT.xtda import XTDA, _so2st, _so2st_matrix, _st2so_matrix
from XTDDFT_dev.utils.backend import backend_info, set_backend


In [3]:
set_backend('CPU')

In [4]:
mol = gto.M(
    atom="""
    C -2.91400807 -0.55823559 0.03052752
    H -2.37721823 -1.66302155 0.09965968
    H -3.96527633 -0.77226144 -0.33224522
    N -2.34030191 0.59563094 0.25221390
    H -2.80790566 1.54696657 0.05511270
    """,
    basis='cc-pvdz',
    unit='A',
    spin=1,
    charge=1,
    verbose=4,
)
mf = dft.ROKS(mol)
mf.xc = 'b3lyp'
mf.conv_tol = 1e-11
mf.conv_tol_grad = 1e-8
mf.max_cycle = 200
#mf.chkfile = 'ROKS.chk'
mf.kernel()

System: uname_result(system='Linux', node='dilandar', release='6.18.33.2-microsoft-standard-WSL2', version='#1 SMP PREEMPT_DYNAMIC Thu Jun 18 21:54:43 UTC 2026', machine='x86_64')  Threads 12
Python 3.11.15 (main, Mar 11 2026, 17:20:07) [GCC 14.3.0]
numpy 2.3.5  scipy 1.17.1  h5py 3.16.0
Date: Wed Sep  9 10:46:54 2026
PySCF version 2.12.1
PySCF path  /home/chang/soft/miniconda3/envs/XSFTDA/lib/python3.11/site-packages/pyscf

[ENV] _GPU4PYSCF_OLD_LD_LIBRARY_PATH /usr/lib/wsl/lib:/home/chang/soft/miniconda3/envs/XSFTDA/lib/python3.11/site-packages/nvidia/cufft/lib:/home/chang/soft/miniconda3/envs/XSFTDA/lib/python3.11/site-packages/nvidia/cuda_runtime/lib:/usr/lib/wsl/lib:/opt/TensorRT-8.6.1.6/lib:
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 5
[INPUT] num. electrons = 15
[INPUT] charge = 1
[INPUT] spin (= nelec alpha-beta = 2S) = 1
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = A
[INPUT] Symbol           X                Y                Z      unit        

np.float64(-94.24491302905811)

In [5]:
XTDA_method = XTDA(mf)

2026-09-09 10:46:56.835 | INFO     | XTDDFT_dev.XTDDFT.base:__init__:809 - Omega:0.0, alpha:0.2, hyb:0.2
2026-09-09 10:46:56.836 | INFO     | XTDDFT_dev.XTDDFT.xtda:__init__:95 - XTDA spin-conserving TDA response


In [6]:
A_so = XTDA_method.get_Amat()

2026-09-09 10:47:08.576 | INFO     | XTDDFT_dev.XTDDFT.xtda:get_Amat:584 - XTDA dense A dimension: 532


In [7]:
ee_so, vv_so = scipy.linalg.eigh(A_so)

In [8]:
A_st = XTDA_method.get_Amat_ST()

2026-09-09 10:47:14.657 | INFO     | XTDDFT_dev.XTDDFT.xtda:get_Amat_ST:680 - XTDA spin-tensor A dimension: 532


In [9]:
ee_st, vv_st = scipy.linalg.eigh(A_st)

In [10]:
np.allclose(ee_so, ee_st)

True

In [11]:
nc = XTDA_method.ctx.nc
no = XTDA_method.ctx.no
nv = XTDA_method.ctx.nv

In [12]:
transform = _so2st_matrix(nc, no, nv)
A_so2st = transform @ A_so @ transform.T

In [13]:
np.allclose(A_st, A_so2st)

True

In [16]:
transform_inv = _st2so_matrix(nc, no, nv)
A_st2so = transform_inv @ A_st @ transform_inv.T

In [17]:
np.allclose(A_so, A_st2so)

True

In [21]:
vv_so2st = _so2st(vv_so, nc, no, nv)
np.diag(np.dot(vv_so2st.T, vv_st))

array([ 1., -1.,  1.,  1., -1., -1.,  1.,  1.,  1., -1., -1.,  1.,  1.,
       -1.,  1., -1., -1.,  1., -1., -1.,  1.,  1., -1.,  1., -1.,  1.,
       -1., -1.,  1., -1., -1.,  1.,  1.,  1.,  1., -1., -1., -1., -1.,
        1., -1.,  1.,  1.,  1., -1.,  1., -1., -1.,  1.,  1., -1.,  1.,
       -1., -1., -1.,  1.,  1., -1.,  1.,  1., -1.,  1.,  1.,  1., -1.,
        1.,  1.,  1.,  1.,  1.,  1., -1., -1., -1., -1.,  1., -1.,  1.,
       -1., -1.,  1.,  1.,  1., -1.,  1.,  1., -1.,  1., -1., -1., -1.,
        1.,  1., -1., -1., -1.,  1., -1., -1.,  1.,  1.,  1., -1.,  1.,
       -1.,  1.,  1., -1.,  1.,  1., -1., -1.,  1., -1., -1., -1., -1.,
        1., -1., -1., -1.,  1.,  1., -1.,  1., -1.,  1.,  1.,  1.,  1.,
        1.,  1., -1., -1., -1., -1.,  1., -1., -1., -1., -1., -1.,  1.,
       -1.,  1., -1.,  1., -1., -1., -1.,  1.,  1.,  1., -1., -1., -1.,
        1., -1., -1., -1.,  1.,  1., -1., -1.,  1.,  1., -1.,  1., -1.,
       -1.,  1., -1., -1., -1., -1.,  1.,  1.,  1., -1., -1., -1